# Load an entity into a Data Vault: TransactionHistory (local)

Loads `TransactionHistory` into a Data Vault on a local Spark session, through DQDs: a hub of
transactions and its satellite, a hub of products, and the link between a transaction and its
product. A hub holds each key once, a link each relationship once, and the satellite takes a new
row for every change of its attributes; nothing is ever updated. Each hub reads a source at its
own key's grain: transactions from the transaction history, products from the product table. Each is one JSON load context; the
Data Vault keys of `scd_context` are described in
[docs/reference/load-context.md](../../docs/reference/load-context.md), and the task in
[docs/guides/load-a-data-vault-entity.md](../../docs/guides/load-a-data-vault-entity.md) and
[docs/guides/load-a-data-vault-link.md](../../docs/guides/load-a-data-vault-link.md).

The notebook runs top to bottom:

1. **Input parameters**: the run switches and the four load contexts.
2. **Library**: `spark_quick_start` from this repository's `src/`, with no install.
3. **Imports**.
4. **Spark session**: local, with Delta.
5. **Pipelines**: the hubs, then the link and the satellite; each result is its cell's output.

It expects a kernel with PySpark and `delta-spark`, started anywhere inside the repository.
The same notebook runs on macOS, Linux and Windows; on Windows it also needs `HADOOP_HOME`
pointing at a Hadoop folder whose `bin/` holds `winutils.exe` and `hadoop.dll`.


## 1. Input parameters

In [ ]:
from pathlib import Path

# The repository root: the nearest folder at or above the working directory holding pyproject.toml.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())


def repo_path(*parts: str) -> str:
    """A path inside the repository, as the load context expects it."""
    return REPO_ROOT.joinpath(*parts).as_posix()


def struct(*fields: tuple[str, str]) -> dict:
    """A Spark StructType as JSON, every field nullable."""
    return {"type": "struct",
            "fields": [{"name": n, "type": t, "nullable": True, "metadata": {}} for n, t in fields]}

In [ ]:
# Run switches
LOAD_TYPE = "Incremental"  # "Full" also records every key absent from the source as deleted
HASH_ALGORITHM = "sha256"  # or "md5"; sealed on each table at its first write
DRIVER_MEMORY = "1g"       # Spark's default; raise it for a large load (backlog F153)
SHUFFLE_PARTITIONS = 200

SOURCE_PARQUET = repo_path("data", "Production_TransactionHistory",
                           "part-00000-f8e129a4-417b-4815-bb81-9c4e9b4b0770-c000.snappy.parquet")
PRODUCT_DELTA = repo_path("data", "Production_Product")
ATTRIBUTES = ["ProductID", "ReferenceOrderID", "ReferenceOrderLineID", "TransactionDate",
              "TransactionType", "Quantity", "ActualCost", "ModifiedDate"]
VAULT = repo_path("data", "Target", "Vault")


def normalised(column: str) -> dict:
    """A business key as every hash key here takes it: cast to string, NULL to "", trimmed, upper-cased."""
    return {"function": "upper", "arguments": [{"function": "trim", "arguments": [
        {"function": "coalesce", "arguments": [
            {"cast": {"expression": {"column": column}, "data_type": "string"}}, {"literal": ""}]}]}]}


def hash_key(name: str, *columns: str) -> dict:
    """A hash key built upstream, the same in every context: sha2 256 of the normalised business
    key, or of several joined by || in a fixed order (a link's key)."""
    joined = normalised(columns[0]) if len(columns) == 1 else {
        "function": "concat_ws", "arguments": [{"literal": "||"}, *(normalised(c) for c in columns)]}
    return {"name": name, "expression": {"function": "sha2", "arguments": [joined, {"literal": 256}]}}


def load_context(style: str, table: str, key: str, selection: list[dict], transformations: list[dict],
                 source: tuple[str, str] = ("parquet", SOURCE_PARQUET), **scd) -> dict:
    """One Data Vault load context: the five required sections and a DQD source."""
    return {
        "global_context": {
            "source_system": "CS",
            "id": table,  # one per context: they share a staging_table_path
            "entity_name": "TransactionHistory",
            "staging_table_path": repo_path("data", "Staging", "Production_TransactionHistory"),
            "target_table_path": f"{VAULT}/{table}",
        },
        "scd_context": {"scd_type": style, "business_key_column": key, "load_type": LOAD_TYPE,
                        "hash_algorithm": HASH_ALGORITHM, **scd},
        "runtime_context": {"application_name": "VaultLoad", "correlation_id": "vault-local",
                            "pipeline_name": table},
        "logging_context": {"log_file_dir": repo_path("data", "Logs"), "logging_level": "INFO",
                            "log_file_name": table, "verbose": False},
        "dqd_context": {"source_kind": source[0], "source_identifier": source[1],
                        "transformations": transformations, "selection": selection},
    }


# The transaction hub: the hash key and the business key as staged, so a reader can check one against the other.
hub_context_dict = load_context("dv_hub", "TransactionHistory_h", "HashKey",
                                [{"name": "HashKey"}, {"name": "TransactionID"}],
                                [hash_key("HashKey", "TransactionID")])

# The product hub, from the product table: one row per product, the grain of its key.
product_hub_context_dict = load_context("dv_hub", "Product_h", "HashKey",
                                        [{"name": "HashKey"}, {"name": "ProductID"}],
                                        [hash_key("HashKey", "ProductID")], source=("delta", PRODUCT_DELTA))

# The link between a transaction and its product: its own key, and each hub's key built as that hub builds it.
link_context_dict = load_context("dv_link", "TransactionHistory_Product_l", "LinkHashKey",
                                 [{"name": "LinkHashKey"}, {"name": "TransactionHashKey"}, {"name": "ProductHashKey"}],
                                 [hash_key("TransactionHashKey", "TransactionID"), hash_key("ProductHashKey", "ProductID"),
                                  hash_key("LinkHashKey", "TransactionID", "ProductID")],
                                 link_hub_key_columns=["TransactionHashKey", "ProductHashKey"])

# The satellite: the hash key and the descriptive attributes, Quantity left out of the hash diff.
satellite_context_dict = load_context("dv_satellite", "TransactionHistory_cs_s", "HashKey",
                                      [{"name": "HashKey"}, *({"name": c} for c in ATTRIBUTES)],
                                      [hash_key("HashKey", "TransactionID")],
                                      exclude_from_hash=["Quantity"])


## 2. Library

In [ ]:
import sys

# The library as a source copy (./src on sys.path), the way it also ships: nothing to install.
# Put first, so an installed copy cannot shadow it. Re-running this cell and the imports cell
# picks up edits to src/ without restarting the kernel.
src = str(REPO_ROOT / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
for name in [m for m in sys.modules if m == "spark_quick_start" or m.startswith("spark_quick_start.")]:
    del sys.modules[name]

import spark_quick_start

print(f"spark_quick_start from {Path(spark_quick_start.__file__).parent}")

## 3. Imports

In [ ]:
import json
import os
import platform
from importlib.metadata import version

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from spark_quick_start.adapters import LocalAdapter
from spark_quick_start.orchestration import dv_hub_pipeline, dv_link_pipeline, dv_satellite_pipeline


## 4. Spark session

In [ ]:
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(hub_context_dict["runtime_context"]["application_name"])
    .master("local[*]")
    # Local mode runs the driver and every task in one JVM, sharing this heap. It is fixed when
    # the JVM starts: a new value needs a kernel restart, not only a new session.
    .config("spark.driver.memory", DRIVER_MEMORY)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

if os.name == "nt":
    # Windows only: Spark reaches the local file system through Hadoop's winutils.exe and
    # hadoop.dll, found in HADOOP_HOME/bin, and must not try Hadoop's native Unix libraries.
    hadoop_home = os.environ.get("HADOOP_HOME")
    if not hadoop_home:
        raise EnvironmentError("Set HADOOP_HOME to a folder whose bin/ holds winutils.exe and hadoop.dll.")
    hadoop_bin = os.path.join(hadoop_home, "bin")
    if hadoop_bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")
    builder = (
        builder
        .config("spark.hadoop.io.native.lib.available", "false")
        .config("spark.driver.extraLibraryPath", hadoop_bin)
        .config("spark.executor.extraLibraryPath", hadoop_bin)
    )

# delta-spark adds the Delta package that matches its own version and this Spark's Scala build
# (delta-spark_2.12 on Spark 3.5, delta-spark_4.x_2.13 on Spark 4), from the Ivy cache when present.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")

print(f"Spark {spark.version} on {platform.system()}, delta-spark {version('delta-spark')}, "
      f"driver memory {spark.sparkContext.getConf().get('spark.driver.memory', '1g')}")

## 5. Pipelines

### The hubs

Run first on every load, so neither the link nor the satellite holds a key its hub lacks.

In [ ]:
adapter = LocalAdapter(base_path=repo_path())

hub_result = dv_hub_pipeline(json.dumps(hub_context_dict), spark, adapter=adapter).run()
product_hub_result = dv_hub_pipeline(json.dumps(product_hub_context_dict), spark, adapter=adapter).run()
hub_result, product_hub_result


### The link

One row per transaction and product pair, ever. A rerun inserts nothing.

In [ ]:
link_result = dv_link_pipeline(json.dumps(link_context_dict), spark, adapter=adapter).run()
link_result


### The satellite

A rerun with nothing changed inserts no rows; change a value in the source and rerun to see one.

In [ ]:
satellite_result = dv_satellite_pipeline(json.dumps(satellite_context_dict), spark, adapter=adapter).run()
satellite_result


### Inspect the satellite: each key's current row

In [ ]:
satellite: DataFrame = spark.read.format("delta").load(satellite_context_dict["global_context"]["target_table_path"])
satellite.createOrReplaceTempView("satellite")
# Open-source Spark 3.5 has no QUALIFY, so the current row is picked in a subquery, as on Fabric.
spark.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY HashKey ORDER BY LoadTS DESC) AS rn FROM satellite
    ) AS s WHERE rn = 1
""").drop("rn").show(5, truncate=False)
